# House Price Prediction — Cleaned Pipeline

In [ ]:
# Data handlingimport pandas as pdimport numpy as np# Modelimport xgboost as xgb# Preprocessingfrom sklearn.impute import SimpleImputerfrom sklearn.preprocessing import OneHotEncoderfrom sklearn.compose import ColumnTransformerfrom sklearn.pipeline import Pipeline# Model selection & evaluationfrom sklearn.model_selection import train_test_split, cross_validatefrom sklearn.metrics import r2_scorefrom sklearn.feature_selection import mutual_info_regression

## 1. Load DataDrop `Id` — it's just a row number, has no relationship to price. Separate target (`y`) from features (`X`).

In [ ]:
df_train = pd.read_csv('house.csv')df_train.drop('Id', axis=1, inplace=True)y = df_train.SalePriceX = df_train.drop(['SalePrice'], axis=1)

Quick look at the data — for us to read, not used by the pipeline.

In [ ]:
pd.set_option('display.max_columns', 85)df_train.head(10)

## 2. Split Into Train/TestDone early on purpose — right after loading, before anything is built or fit. This makes it visually obvious that nothing downstream can ever touch `X_test`/`y_test` until the final evaluation step.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

## 3. Identify Column TypesNeeded so we know which columns get numeric treatment vs categorical treatment.

In [ ]:
numeric_cols = X.select_dtypes(include=['int64', 'float64']).columnscategorical_cols = X.select_dtypes(include=['object']).columns

## 4. Preprocessing Steps- Numeric: fill missing values with the median.- Categorical: fill missing values with `'None'`, then one-hot encode. Two steps chained in their own mini `Pipeline`, since a `ColumnTransformer` branch only accepts one tool — `handle_unknown='ignore'` stops the encoder from crashing if `X_test` has a category `X_train` never saw.

In [ ]:
numeric_transformer = SimpleImputer(strategy='median')categorical_transformer = Pipeline(steps=[    ('impute', SimpleImputer(strategy='constant', fill_value='None')),    ('onehot', OneHotEncoder(handle_unknown='ignore'))])

## 5. Combine Into One Pipeline`ColumnTransformer` routes each column group to its matching preprocessing step. `Pipeline` then chains that preprocessor with the model, so one `.fit()` call handles preprocessing AND training correctly, fit only on train — this is what makes the leakage bug from earlier structurally impossible now.

In [ ]:
preprocessor = ColumnTransformer(transformers=[    ('num', numeric_transformer, numeric_cols),    ('cat', categorical_transformer, categorical_cols)])model = xgb.XGBRegressor(n_estimators=100, learning_rate=0.1)full_pipeline = Pipeline(steps=[    ('preprocessor', preprocessor),    ('model', model)])

## 6. Cross-ValidateTrains/tests 5 fresh copies of `full_pipeline` on 5 different slices of `X_train` only — gives an honest average error estimate before we ever touch `X_test`.

In [ ]:
cv_results = cross_validate(full_pipeline, X_train, y_train, cv=5, scoring='neg_mean_absolute_error')cv_results

## 7. Fit, Predict, Evaluate`cross_validate` never leaves `full_pipeline` trained (it only ever trains throwaway copies) — so we still fit it ourselves here before predicting.

In [ ]:
full_pipeline.fit(X_train, y_train)predict = full_pipeline.predict(X_test)r2 = r2_score(y_test, predict)print(f"R2 Score: {r2:.3f}")

## 8. Feature Importance — Mutual InformationRun on `X_train`/`y_train` (features vs. real target — not predictions), numeric columns only, gaps filled first. Ranks which raw columns carry the most signal about price.

In [ ]:
numeric_data = X_train[numeric_cols].fillna(X_train[numeric_cols].median())mi_scores = mutual_info_regression(numeric_data, y_train)mi_scores = pd.Series(mi_scores, index=numeric_cols, name='MI Scores').sort_values(ascending=False)mi_scores.nlargest(10)